In [1]:
import pandas as pd
import re

POSITIVE_WORDS = {
    "good",
    "great",
    "excellent",
    "amazing",
    "awesome",
    "fantastic",
    "nice",
    "love",
    "well",
    "happy",
    "wonderful",
    "positive",
    "best",
    "perfect",
    "sweet",
    "cool",
    "liked",
    "liked",
    "nice",
    "pleased",
}
NEGATIVE_WORDS = {
    "bad",
    "terrible",
    "awful",
    "worst",
    "hate",
    "poor",
    "sad",
    "negative",
    "horrible",
    "disappointed",
    "unhappy",
    "sucks",
    "disgusting",
    "annoying",
    "lame",
    "boring",
}
INTENSIFIERS = {"very", "really", "extremely", "so", "quite", "too", "not"}

train_path = "../input/tweet-sentiment-extraction/train.csv"
train = pd.read_csv(train_path)


def _build_context_stats(df):
    left_max = {}
    right_max = {}
    for _, row in df.iterrows():
        text = row["text"]
        selected = row["selected_text"]
        sentiment = row["sentiment"].lower()
        if sentiment == "neutral":
            continue
        lexicon = POSITIVE_WORDS if sentiment == "positive" else NEGATIVE_WORDS

        sel_tokens = re.findall(r"\b\w+\b", selected.lower())
        word = None
        for w in sel_tokens:
            if w in lexicon:
                word = w
                break
        if not word:
            continue

        word_idx = sel_tokens.index(word)
        left = word_idx
        right = len(sel_tokens) - word_idx - 1

        left_max[word] = max(left_max.get(word, 0), left)
        right_max[word] = max(right_max.get(word, 0), right)

    return left_max, right_max


LEFT_MAX, RIGHT_MAX = _build_context_stats(train)

extra_pos = set()
extra_neg = set()
for _, row in train.iterrows():
    sentiment = row["sentiment"].lower()
    if sentiment not in {"positive", "negative"}:
        continue
    tokens = re.findall(r"\b\w+\b", row["selected_text"].lower())
    if sentiment == "positive":
        extra_pos.update(tokens)
    else:
        extra_neg.update(tokens)

POSITIVE_WORDS = POSITIVE_WORDS.union(extra_pos)
NEGATIVE_WORDS = NEGATIVE_WORDS.union(extra_neg)


def _build_phrase_set(df, sentiment):
    phrases = set()
    for _, row in df.iterrows():
        if row["sentiment"].lower() != sentiment:
            continue
        txt = row["selected_text"].strip().lower()
        tokens = re.findall(r"\b\w+(?:'\w+)?\b", txt)
        n = len(tokens)
        for i in range(n):
            for j in range(i, min(i + 4, n)):
                start = txt.find(tokens[i])
                end = txt.find(tokens[j], start) + len(tokens[j])
                phrases.add(txt[start:end])
    return phrases


POSITIVE_PHRASES = _build_phrase_set(train, "positive")
NEGATIVE_PHRASES = _build_phrase_set(train, "negative")




In [2]:
def extract_selected_text(text: str, sentiment: str) -> str:
    """
    Heuristic with phrase‑lookup fallback:
    - Neutral → whole tweet.
    - Positive/Negative → first try to find the longest phrase (1‑4 tokens) from the
      training phrase set that occurs verbatim in the tweet. If found, return it.
    - If no phrase matches, fall back to the token‑expansion heuristic.
    """
    if sentiment.lower() == "neutral":
        return text.strip()

    phrase_set = (
        POSITIVE_PHRASES if sentiment.lower() == "positive" else NEGATIVE_PHRASES
    )
    lowered_text = text.lower()
    best_phrase = ""
    token_pattern = r"\b\w+(?:'\w+)?\b"
    tokens = list(re.finditer(token_pattern, text))
    n_tokens = len(tokens)
    for start in range(n_tokens):
        for end in range(start, min(start + 4, n_tokens)):
            span_text = text[tokens[start].start() : tokens[end].end()]
            if span_text.lower() in phrase_set:
                if len(span_text) > len(best_phrase):
                    best_phrase = span_text

    if best_phrase:
        end_idx = text.find(best_phrase) + len(best_phrase)
        while end_idx < len(text) and text[end_idx] in ",.!?;:":
            end_idx += 1
        return text[text.find(best_phrase) : end_idx].strip()

    sentiment_lc = sentiment.lower()
    lexicon = POSITIVE_WORDS if sentiment_lc == "positive" else NEGATIVE_WORDS

    match_indices = []
    for i, m in enumerate(tokens):
        w = m.group().lower()
        if w in lexicon:
            match_indices.append(i)
        elif sentiment_lc == "negative" and w == "not" and i + 1 < len(tokens):
            nxt = tokens[i + 1].group().lower()
            if nxt in POSITIVE_WORDS:
                match_indices.append(i)  # start span at the negation word

    if not match_indices:
        return text.strip()

    best_span = None
    best_len = -1

    for idx in match_indices:
        word = tokens[idx].group().lower()

        # ----- left expansion: only intensifiers, limited by a modest learned size -----
        start_idx = idx
        left_expanded = 0
        left_limit = LEFT_MAX.get(word, 3)  # smaller default to avoid over‑capture
        while start_idx > 0:
            prev_word = tokens[start_idx - 1].group().lower()
            if prev_word in INTENSIFIERS and left_expanded < left_limit:
                start_idx -= 1
                left_expanded += 1
            else:
                break

        # ----- right expansion: stop at punctuation, limited by a modest learned size -----
        end_idx = idx
        right_expanded = 0
        right_limit = RIGHT_MAX.get(word, 5)  # smaller default for tighter spans
        while (end_idx + 1) < len(tokens) and right_expanded < right_limit:
            next_char_pos = tokens[end_idx].end()
            if next_char_pos < len(text) and text[next_char_pos] in ",.!?;:":
                break
            end_idx += 1
            right_expanded += 1

        span_len = end_idx - start_idx + 1
        if span_len > best_len:
            best_len = span_len
            best_span = (tokens[start_idx].start(), tokens[end_idx].end())

    if best_span:
        start_char, end_char = best_span
        while end_char < len(text) and text[end_char] in ",.!?;:":
            end_char += 1
        while start_char > 0 and text[start_char - 1] in ",.!?;:":
            start_char -= 1
        return text[start_char:end_char].strip()

    return text.strip()




In [3]:
test_path = "../input/tweet-sentiment-extraction/test.csv"
test = pd.read_csv(test_path)

test["selected_text"] = test.apply(
    lambda row: extract_selected_text(row["text"], row["sentiment"]), axis=1
)

submission_path = "submission.csv"
test[["textID", "selected_text"]].to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
